# 03 — Entrenar, validar y exportar

1. **Linear** `linear_v1.json` — softmax CE, producción sin torch.
2. **MLP** `mlp_v1` — `Linear(35,64)→ReLU→Linear(64,1)`, un `.pt` por régimen.
3. **Validación** vs heurístico y vs placeholder, mismo `packing_mode=online`.
4. Smoke de producto: 1 pedido de los 5 demo con `LearnedPlacementPolicy.from_path`.

Hiperparámetros solo sobre `data/val`. Los 5 demo no se usan para tunear.

In [2]:
from pathlib import Path
import sys

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")

REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

print("ML_ROOT =", ML_ROOT)

ML_ROOT = /home/edmundo/packing-services/online_policy_ml


In [3]:
from config import HIDDEN_SIZE, HEURISTIC_P3S2_MAX_ORDERS, MLP_EPOCHS, SEED, mlp_lr
from paths import (
    BED_JSON_NAME,
    HOLDOUT_DIR,
    ORDER_IDS_NAME,
    PLACEHOLDER_LINEAR,
    REPORTS_DIR,
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    model_path_linear,
    model_path_mlp,
    transitions_path,
)
from splits import load_orders
from collect import load_transitions
from train_linear import fit_linear, linear_accuracy
from train_mlp import fit_mlp
from train_mlp_numpy import arrays_to_torch_state, fit_mlp_numpy
from export_ckpt import export_linear_json, export_mlp_pt
from evaluate import evaluate_orders, smoke_from_path, write_report
from packing_services.online.learned.policy import LearnedPlacementPolicy
from packing_services.online.features import FEATURE_DIM, FEATURE_NAMES
from packing_services.datasets.bed_bpp import smallest_order_id

for split, p, s in (("train",1,1),("val",1,1),("train",3,2),("val",3,2)):
    assert transitions_path(split, p, s).is_file(), f"Falta fase 2: {transitions_path(split, p, s)}"

train_p1 = load_transitions(transitions_path("train", 1, 1))
val_p1 = load_transitions(transitions_path("val", 1, 1))
train_p3 = load_transitions(transitions_path("train", 3, 2))
val_p3 = load_transitions(transitions_path("val", 3, 2))
orders_val = load_orders(VAL_DIR / BED_JSON_NAME)
orders_test = load_orders(TEST_DIR / BED_JSON_NAME)
ids_val = json.loads((VAL_DIR / ORDER_IDS_NAME).read_text(encoding="utf-8"))
ids_test = json.loads((TEST_DIR / ORDER_IDS_NAME).read_text(encoding="utf-8"))
holdout = load_orders(HOLDOUT_DIR / "5_bed-bpp.json")

print("train p1s1", train_p1["n_transitions"], "| val", val_p1["n_transitions"])
print("train p3s2", train_p3["n_transitions"], "| val", val_p3["n_transitions"])
print("placeholder", PLACEHOLDER_LINEAR.is_file())

train p1s1 314 | val 109
train p3s2 315 | val 112
placeholder True


## Linear v1 (régimen p=1 s=1)

In [5]:
lin = fit_linear(train_p1["transitions"], epochs=8, lr=3e-4, l2=1e-5, seed=SEED)
acc_tr = linear_accuracy(train_p1["transitions"], lin["weights"], lin["bias"])
acc_va = linear_accuracy(val_p1["transitions"], lin["weights"], lin["bias"])
linear_path = model_path_linear()
export_linear_json(
    lin["weights"],
    lin["bias"],
    linear_path,
    notes=f"BC softmax-CE P2O {train_p1['teacher']} seed={SEED} p=1 s=1",
)
doc = json.loads(linear_path.read_text(encoding="utf-8"))
assert doc["format"] == "packing-services-online-policy"
assert doc["feature_names"] == list(FEATURE_NAMES)
assert len(doc["weights"]) == FEATURE_DIM
print("acc train", round(acc_tr, 3), "acc val", round(acc_va, 3))
print("export", linear_path)
pd.DataFrame(lin["history"]).plot(x="epoch", y=["loss", "accuracy"], figsize=(6, 3), title="Linear BC")
plt.tight_layout()
plt.show()

acc train 0.997 acc val 0.991
export /home/edmundo/packing-services/online_policy_ml/artifacts/models/linear_v1.json


03_entrenar_validar_exportar.ipynb:cell4:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## MLP mlp_v1

Arquitectura exacta del loader (`Linear(35,64)→ReLU→Linear(64,1)`). El mejor epoch se guarda por **val acc** (empate: menor val loss). `p=3,s=2` usa `lr=3e-4` y Adam con weight decay; la capa de salida arranca en ceros.

Si no hay torch, se entrena en NumPy y el `.pt` solo se escribe si torch está instalado.

In [7]:
has_torch = True
try:
    import torch
    print("torch", torch.__version__)
except ImportError:
    has_torch = False
    print("Sin torch en el import inicial. Se entrena en NumPy.")

mlp_paths = {}
mlp_histories = {}
mlp_meta = {}
for (p, s), tr, va in (((1, 1), train_p1, val_p1), ((3, 2), train_p3, val_p3)):
    print(f"\n--- MLP p={p} s={s} hidden={HIDDEN_SIZE} ---")
    state = None
    if has_torch:
        fit = fit_mlp(
            tr["transitions"], va["transitions"],
            hidden_size=HIDDEN_SIZE, epochs=MLP_EPOCHS, lr=mlp_lr(p, s), seed=SEED,
        )
        state = fit["state_dict"]
        mlp_histories[(p, s)] = fit["history"]
        mlp_meta[(p, s)] = {
            "lr": mlp_lr(p, s),
            "best_epoch": int(fit["best_epoch"]),
            "best_val_acc": fit["best_val_acc"],
            "best_val_loss": fit["best_val_loss"],
            "select_best": fit["select_best"],
        }
        print(f"  lr={mlp_lr(p, s)} best_epoch={fit['best_epoch']} val_acc={fit['best_val_acc']:.4f} val_loss={fit['best_val_loss']:.4f}")
    else:
        fitn = fit_mlp_numpy(
            tr["transitions"], va["transitions"],
            hidden_size=HIDDEN_SIZE, epochs=MLP_EPOCHS, lr=mlp_lr(p, s), seed=SEED,
        )
        mlp_histories[(p, s)] = fitn["history"]
        mlp_meta[(p, s)] = {
            "lr": mlp_lr(p, s),
            "best_epoch": int(fitn.get("best_epoch") or 0),
            "best_val_acc": fitn.get("best_val_acc"),
            "backend": "numpy",
        }
        try:
            state = arrays_to_torch_state(fitn["state_arrays"])
            has_torch = True
        except ImportError:
            state = None
    print(pd.DataFrame(mlp_histories[(p, s)]).round(4).tail(3).to_string(index=False))
    if state is not None:
        path = model_path_mlp(p, s)
        export_mlp_pt(state, path, hidden_size=HIDDEN_SIZE)
        mlp_paths[(p, s)] = path
        print("export", path)
    else:
        print("sin .pt (instala torch para exportar mlp_v1)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, key in zip(axes, ((1, 1), (3, 2))):
    hist = pd.DataFrame(mlp_histories[key])
    ax.plot(hist["epoch"], hist["train_acc"], label="train acc")
    if "val_acc" in hist:
        ax.plot(hist["epoch"], hist["val_acc"], label="val acc")
    ax.set_title(f"MLP p={key[0]} s={key[1]}")
    ax.set_xlabel("epoch")
    ax.legend()
axes[0].set_ylabel("accuracy")
fig.tight_layout()
plt.show()

torch 2.14.0+cpu

--- MLP p=1 s=1 hidden=64 ---
  lr=0.001 best_epoch=5 val_acc=0.9908 val_loss=0.4240
 epoch  train_loss  train_acc  val_loss  val_acc
   8.0      0.0351     0.9968    6.8416   0.9174
   9.0      0.1966     0.9968    0.6085   0.9908
  10.0      0.0238     1.0000    0.5512   0.9908
export /home/edmundo/packing-services/online_policy_ml/artifacts/models/mlp_v1_p1s1.pt

--- MLP p=3 s=2 hidden=64 ---
  lr=0.0003 best_epoch=4 val_acc=0.9286 val_loss=9.7793
 epoch  train_loss  train_acc  val_loss  val_acc
   8.0     53.1091     0.8698   14.2894   0.9196
   9.0     46.4018     0.8571   38.4478   0.9286
  10.0     77.8356     0.8540   32.3431   0.9196
export /home/edmundo/packing-services/online_policy_ml/artifacts/models/mlp_v1_p3s2.pt


03_entrenar_validar_exportar.ipynb:cell6:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Validación packing

Mismo `packing_mode=online`. El heurístico `p=3,s=2` se limita a pocos pedidos (lookahead caro).

In [9]:
def pack_table(orders, order_ids, lookahead_p, select_s, max_orders=None):
    rows = []
    rows += evaluate_orders(orders, order_ids, engine="heuristic", lookahead_p=lookahead_p, select_s=select_s, max_orders=max_orders)
    rows += evaluate_orders(orders, order_ids, engine="learned", lookahead_p=lookahead_p, select_s=select_s, model_path=str(PLACEHOLDER_LINEAR), max_orders=max_orders)
    rows += evaluate_orders(orders, order_ids, engine="learned", lookahead_p=lookahead_p, select_s=select_s, model_path=str(linear_path), max_orders=max_orders)
    if (lookahead_p, select_s) in mlp_paths:
        rows += evaluate_orders(orders, order_ids, engine="learned", lookahead_p=lookahead_p, select_s=select_s, model_path=str(mlp_paths[(lookahead_p, select_s)]), max_orders=max_orders)
    return rows

def pretty(rows):
    df = pd.DataFrame(rows)
    df["model"] = df.apply(
        lambda r: r["engine"] if r["engine"] == "heuristic" else Path(str(r["model_path"])).name,
        axis=1,
    )
    return df.groupby("model", as_index=False).agg(
        n=("order_id", "count"),
        util=("volume_utilization", "mean"),
        packed=("items_packed", "mean"),
        valid=("is_valid", "mean"),
        sec=("seconds", "mean"),
    ).round(4)

print("val p=1 s=1 ...")
rows_val_p1 = pack_table(orders_val, ids_val, 1, 1)
print("test p=1 s=1 ...")
rows_test_p1 = pack_table(orders_test, ids_test, 1, 1)
print("val p=3 s=2 (subset) ...")
rows_val_p3 = pack_table(orders_val, ids_val, 3, 2, max_orders=HEURISTIC_P3S2_MAX_ORDERS)

print("\nVAL p=1 s=1"); display(pretty(rows_val_p1))
print("TEST p=1 s=1"); display(pretty(rows_test_p1))
print("VAL p=3 s=2 (subset)"); display(pretty(rows_val_p3))
assert pretty(rows_val_p1)["valid"].min() == 1.0
assert pretty(rows_test_p1)["valid"].min() == 1.0

val p=1 s=1 ...
test p=1 s=1 ...
val p=3 s=2 (subset) ...

VAL p=1 s=1
                          model  n    util  packed  valid     sec
0                     heuristic  8  0.6770  13.625    1.0  0.0461
1                linear_v1.json  8  0.6838  13.750    1.0  0.0559
2                mlp_v1_p1s1.pt  8  0.6838  13.750    1.0  0.0672
3  online_policy_linear_v1.json  8  0.6770  13.625    1.0  0.0542
TEST p=1 s=1
                          model  n    util  packed  valid     sec
0                     heuristic  8  0.6716  13.000    1.0  0.0474
1                linear_v1.json  8  0.6793  13.125    1.0  0.0471
2                mlp_v1_p1s1.pt  8  0.6793  13.125    1.0  0.0583
3  online_policy_linear_v1.json  8  0.6716  13.000    1.0  0.0490
VAL p=3 s=2 (subset)
                          model  n    util   packed  valid     sec
0                     heuristic  3  0.6924  13.6667    1.0  3.1860
1                linear_v1.json  3  0.6738  13.3333    1.0  0.1276
2                mlp_v1_p3s2.pt  3

## Smoke de producto (1 pedido de los 5 demo, sin tunear)

In [11]:
smoke_id = smallest_order_id(holdout)
print("holdout (más pequeño):", smoke_id)
candidates = [("linear", linear_path)]
for key, name in (((1, 1), "mlp_p1s1"), ((3, 2), "mlp_p3s2")):
    if key in mlp_paths:
        candidates.append((name, mlp_paths[key]))

smoke_rows = []
for name, path in candidates:
    policy = LearnedPlacementPolicy.from_path(path)
    del policy
    row = smoke_from_path(path, holdout, smoke_id)
    row["checkpoint"] = name
    smoke_rows.append(row)
    print(name, "valid", row["is_valid"], "packed", row["items_packed"], "util", round(row["volume_utilization"], 4))
    assert row["is_valid"], f"{name} inválido en holdout {smoke_id}"

display(pd.DataFrame(smoke_rows)[["checkpoint", "order_id", "volume_utilization", "items_packed", "is_valid", "seconds"]])

holdout (más pequeño): 00100408
linear valid True packed 26 util 0.6464
mlp_p1s1 valid True packed 26 util 0.6464
mlp_p3s2 valid True packed 26 util 0.6464
  checkpoint  order_id  volume_utilization  items_packed  is_valid   seconds
0     linear  00100408            0.646376            26      True  0.671900
1   mlp_p1s1  00100408            0.646376            26      True  0.687303
2   mlp_p3s2  00100408            0.646376            26      True  0.507893


In [12]:
prod_path = mlp_paths.get((1, 1), linear_path)
report = {
    "fase": 3,
    "seed": SEED,
    "hidden_size": HIDDEN_SIZE,
    "teacher": train_p1["teacher"],
    "linear": str(linear_path),
    "mlp": {f"p{p}s{s}": str(path) for (p, s), path in mlp_paths.items()},
    "bc_linear_acc": {"train": acc_tr, "val": acc_va},
    "mlp_train": {f"p{p}s{s}": meta for (p, s), meta in mlp_meta.items()},
    "val_p1s1": pretty(rows_val_p1).to_dict(orient="records"),
    "test_p1s1": pretty(rows_test_p1).to_dict(orient="records"),
    "val_p3s2_subset": pretty(rows_val_p3).to_dict(orient="records"),
    "smoke_holdout": smoke_rows,
    "production": {
        "model_path": str(prod_path),
        "packing_mode": "online",
        "lookahead_p": 1,
        "select_s": 1,
        "note": "Si usas mlp_v1_p3s2.pt pon lookahead_p=3 y select_s=2",
    },
}
write_report(REPORTS_DIR / "03_validacion.json", report)
print("\n=== Línea para el chat de producto ===")
print(f'parameters.model_path = "{prod_path}"')
print("packing_mode=online, lookahead_p=1, select_s=1")
print("reporte", REPORTS_DIR / "03_validacion.json")


=== Línea para el chat de producto ===
parameters.model_path = "/home/edmundo/packing-services/online_policy_ml/artifacts/models/mlp_v1_p1s1.pt"
packing_mode=online, lookahead_p=1, select_s=1
reporte /home/edmundo/packing-services/online_policy_ml/artifacts/reports/03_validacion.json


Checkpoints en `online_policy_ml/artifacts/models/`. El execute del repo ya sabe cargarlos.